In [1]:
import math
import time
from dataclasses import dataclass

import numpy as np
from scipy.interpolate import interp1d
from tdmclient import ClientAsync

# ------------------------------------------------------------------
# IMPORT YOUR CALIBRATION / SENSOR GEOMETRY
# ------------------------------------------------------------------
# In your real project, you should already have these defined in local_occupancy.py
#   - sensor_measurements: 1D array of raw Thymio prox values
#   - sensor_distances:    1D array of corresponding distances in cm
#   - sensor_pos_from_center: list of (x_forward_cm, y_left_cm) sensor positions
#   - sensor_angles:          list of sensor angles (rad) in robot forward/left frame

from local_occupancy import (
        sensor_measurements,
        sensor_distances,
        sensor_pos_from_center,
        sensor_angles,
    )



# ------------------------------------------------------------------
# BASIC UTILITIES
# ------------------------------------------------------------------

def wrap_to_pi(angle: float) -> float:
    """Wrap any angle (rad) to (-pi, pi]."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def sensor_val_to_cm_dist(val: int) -> float:
    """
    Convert a Thymio prox value to a distance in cm using your calibration
    (sensor_measurements -> sensor_distances).
    """
    if val == 0:
        return np.inf
    f = interp1d(sensor_measurements, sensor_distances)
    return float(f(val))


def obstacles_pos_from_sensor_vals(sensor_vals):
    """
    Convert Thymio's horizontal proximity readings to obstacle positions
    in the ROBOT frame, in centimeters.

    Robot frame convention:
        x_left    > 0 -> obstacle on the LEFT
        x_left    < 0 -> obstacle on the RIGHT
        y_forward > 0 -> obstacle in FRONT
    """
    dists_cm = [sensor_val_to_cm_dist(v) for v in sensor_vals]

    obstacles = []
    # sensor_pos_from_center entries are (sx_forward_cm, sy_left_cm)
    # angles are in the forward/left frame.
    for (sx_forward, sy_left), d, a in zip(sensor_pos_from_center, dists_cm, sensor_angles):
        if not np.isfinite(d):
            continue

        # In "forward/left" frame
        obs_forward = sx_forward + d * math.cos(a)
        obs_left = sy_left + d * math.sin(a)

        # Convert to our robot frame (x_left, y_forward)
        x_left = obs_left
        y_forward = obs_forward

        obstacles.append([x_left, y_forward])

    return np.array(obstacles, dtype=float)


# ------------------------------------------------------------------
# LOCAL OCCUPANCY GRID
# ------------------------------------------------------------------

class LocalOccupancyGrid:
    """
    Robot-centred occupancy grid.

    Frame:
        x: left  (+), right (-)
        y: front (+), back  (-)
    """

    def __init__(
        self,
        size_m: float = 0.8,
        resolution_m: float = 0.01,
        hit_inc: float = 0.4,
        decay: float = 0.85,
        robot_radius_m: float = 0.12,
    ):
        """
        size_m        : physical size of grid (square) in meters
        resolution_m  : cell size [m]
        hit_inc       : occupancy increment when a cell is hit
        decay         : multiplicative decay each update (0..1)
        robot_radius_m: radius used for obstacle inflation
        """
        self.size_m = size_m
        self.resolution = resolution_m
        self.hit_inc = hit_inc
        self.decay = decay
        self.robot_radius_m = robot_radius_m

        self.half_size = size_m / 2.0
        self.n_cells = int(size_m / resolution_m)
        self.grid = np.zeros((self.n_cells, self.n_cells), dtype=float)

    # -------------- indexing helpers --------------

    def _point_to_index(self, x_m, y_m):
        """Convert a point in robot frame [m] to grid indices (iy, ix)."""
        if abs(x_m) > self.half_size or abs(y_m) > self.half_size:
            return None
        ix = int((x_m + self.half_size) / self.resolution)
        iy = int((y_m + self.half_size) / self.resolution)
        ix = min(max(ix, 0), self.n_cells - 1)
        iy = min(max(iy, 0), self.n_cells - 1)
        return iy, ix

    def _index_to_point(self, iy, ix):
        """Convert grid indices (iy, ix) back to a point in robot frame [m]."""
        x_m = (ix + 0.5) * self.resolution - self.half_size
        y_m = (iy + 0.5) * self.resolution - self.half_size
        return x_m, y_m

    # -------------- update from sensors --------------

    def update_from_sensor_vals(self, sensor_vals):
        """
        Update occupancy from Thymio horizontal proximity readings.

        - First decays all cells.
        - Then adds hits around each detected obstacle.
        """
        # decay old values
        self.grid *= self.decay

        # convert current sensor readings to obstacle positions in robot frame
        obs_cm = obstacles_pos_from_sensor_vals(sensor_vals)

        inflate_cells = max(1, int(self.robot_radius_m / self.resolution))

        for ox_cm, oy_cm in obs_cm:
            if not np.isfinite(ox_cm) or not np.isfinite(oy_cm):
                continue

            x_m = ox_cm / 100.0
            y_m = oy_cm / 100.0

            idx = self._point_to_index(x_m, y_m)
            if idx is None:
                continue

            iy, ix = idx

            # Inflate in a disk around (iy, ix)
            for dy in range(-inflate_cells, inflate_cells + 1):
                for dx in range(-inflate_cells, inflate_cells + 1):
                    jy = iy + dy
                    jx = ix + dx
                    if 0 <= jy < self.n_cells and 0 <= jx < self.n_cells:
                        if dx * dx + dy * dy <= inflate_cells * inflate_cells:
                            self.grid[jy, jx] = min(
                                1.0, self.grid[jy, jx] + self.hit_inc
                            )


# ------------------------------------------------------------------
# LOCAL NAVIGATOR (PURE AVOIDANCE)
# ------------------------------------------------------------------

@dataclass
class LocalNavConfig:
    occ_threshold: float = 0.3
    rep_influence_radius: float = 0.8      # how far obstacles "repel" [m]
    k_att: float = 1.2                     # forward attraction magnitude
    k_rep: float = 0.85                    # repulsion gain


class LocalNavigator:
    def __init__(self, grid: LocalOccupancyGrid, cfg: LocalNavConfig):
        self.grid = grid
        self.cfg = cfg

    def _repulsive_vector(self):
        """
        Compute repulsive force in ROBOT frame [x_left, y_forward].

        - Only cells with occupancy > occ_threshold are considered.
        - Only cells in FRONT (y_forward > 0) repel.
        - Direction is biased to be more LATERAL than purely backwards.
        - Final norm is saturated so it doesn't explode.
        """
        F_rep = np.zeros(2, dtype=float)
        R = self.cfg.rep_influence_radius
        eps = 1e-3

        occ_idxs = np.argwhere(self.grid.grid > self.cfg.occ_threshold)

        for iy, ix in occ_idxs:
            occ_val = float(self.grid.grid[iy, ix])  # in [0,1]
            x_m, y_m = self.grid._index_to_point(iy, ix)
            d = math.sqrt(x_m * x_m + y_m * y_m)

            # ignore behind robot
            if y_m <= 0.0:
                continue
            if d < eps or d > R:
                continue

            # --- more lateral repulsion ---
            # vector from obstacle to robot (lat, back)
            lat = -x_m / d      # left/right
            back = -y_m / d     # backwards

            # bias to keep lateral strong, reduce backward component
            dir_vec = np.array([lat, 0.4 * back], dtype=float)

            # re-normalise direction
            n = np.linalg.norm(dir_vec)
            if n < 1e-6:
                continue
            dir_vec /= n

            # distance-based weight (stronger when closer)
            s = max(0.0, (R - d) / R)
            s = s * s

            mag = self.cfg.k_rep * occ_val * s
            F_rep += mag * dir_vec

        # ---- SATURATE F_rep magnitude ----
        max_F_rep = 3.0 * self.cfg.k_att  # allow repulsion up to 3x attraction
        norm_rep = np.linalg.norm(F_rep)
        if norm_rep > max_F_rep:
            F_rep = F_rep * (max_F_rep / norm_rep)

        return F_rep

    def compute_avoidance_direction_robot(self, sensor_vals):
        """
        PURE LOCAL AVOIDANCE in ROBOT frame.

        Returns:
            F_att: np.array([x_left, y_forward]) attractive (always forward)
            F_rep: repulsive vector from occupancy
            F_tot: sum of both (desired direction)
        """
        # Update occupancy grid from current readings
        self.grid.update_from_sensor_vals(sensor_vals)

        # Attractive: go straight ahead (y_forward)
        F_att = np.array([0.0, self.cfg.k_att], dtype=float)

        # Repulsive from occupancy grid
        F_rep = self._repulsive_vector()

        # If nothing close in raw sensors, ignore repulsion
        if sensor_vals is None or max(sensor_vals) < 300:
            F_rep = np.zeros(2, dtype=float)

        F_tot = F_att + F_rep

        # Avoid exact zero vector
        if np.linalg.norm(F_tot) < 1e-6:
            F_tot = np.array([0.0, 1.0], dtype=float)

        return F_att, F_rep, F_tot


# ------------------------------------------------------------------
# GO-TO-GOAL CONTROLLER (ONLY FOR WHEEL MAPPING HERE)
# ------------------------------------------------------------------

@dataclass
class GoToGoalGains:
    Komega: float = 2.0    # TURNING GAIN (was 1.0 -> now more aggressive)
    v_max: float = 0.35    # max forward speed [m/s]
    w_max: float = 4.0     # max angular speed [rad/s]


@dataclass
class ThymioKinematics:
    wheel_radius: float = 0.022
    half_axle: float = 0.0475
    motor_gain: float = 10.0
    motor_max: int = 300


class GoToGoalController:
    def __init__(self, gains: GoToGoalGains, kin: ThymioKinematics):
        self.g = gains
        self.kin = kin

    def unicycle_to_wheels(self, v, w):
        """
        Map unicycle commands (v, w) to Thymio wheel target speeds.

        v : linear speed [m/s]
        w : angular speed [rad/s], positive -> turn left
        """
        r = self.kin.wheel_radius
        L = self.kin.half_axle

        # wheel angular speeds [rad/s]
        phi_L = (v / r) - (L / r) * w
        phi_R = (v / r) + (L / r) * w

        # map to motor commands
        uL = int(self.kin.motor_gain * phi_L)
        uR = int(self.kin.motor_gain * phi_R)

        # saturate motor commands
        uL = max(-self.kin.motor_max, min(self.kin.motor_max, uL))
        uR = max(-self.kin.motor_max, min(self.kin.motor_max, uR))

        return uL, uR


# ------------------------------------------------------------------
# SMALL DEBUG HELPERS
# ------------------------------------------------------------------

def vec_to_polar(v):
    """
    Convert a robot-frame vector [x_left, y_forward] into (magnitude, angle_deg).

    angle_deg:
        0°  -> straight ahead
        >0° -> steer left
        <0° -> steer right
    """
    x_left, y_forward = v
    mag = float(np.linalg.norm(v))
    ang_deg = math.degrees(math.atan2(x_left, y_forward))
    return mag, ang_deg


# ------------------------------------------------------------------
# AVOIDANCE: VECTOR -> (v, w) + HEURISTICS
# ------------------------------------------------------------------

def avoidance_unicycle_cmd(F_tot, gains: GoToGoalGains, v_far: float = None, v_turn: float = 0.10):
    """
    Map a desired direction vector in ROBOT frame (x_left, y_forward)
    to unicycle (v, w) using GoToGoalGains.

    - For small angles -> drive forward fast.
    - For medium angles -> slower forward, strong turn.
    - For large angles -> pivot in place (v ~ 0).
    """
    x_left, y_forward = F_tot
    norm = float(np.linalg.norm(F_tot) + 1e-6)

    if v_far is None:
        v_far = gains.v_max  # e.g. 0.35

    # heading angle relative to forward:
    alpha = math.atan2(x_left, y_forward)  # + = turn left, - = turn right
    angle_deg = math.degrees(alpha)

    # baseline forward component in [0, 1]
    forward_component = max(0.0, y_forward / norm)

    # --- piecewise behaviour depending on how sideways we want to go ---
    if abs(angle_deg) > 70.0:
        # almost sideways/backwards: pivot in place
        v = 0.0
    elif abs(angle_deg) > 35.0:
        # turning strongly: go slowly forward
        v = v_turn * forward_component
    else:
        # mostly forward: go faster
        v = v_far * forward_component

    # angular speed from angle
    w = gains.Komega * alpha

    # clip to controller limits
    v = max(-gains.v_max, min(gains.v_max, v))
    w = max(-gains.w_max, min(gains.w_max, w))

    return v, w


def apply_side_heuristic(sensor_vals, w, min_turn=1.5, side_thr=1200):
    """
    Enforce the sign and MINIMUM magnitude of w based on which outer
    front sensors see an obstacle.

    Thymio prox.horizontal indices:
        0: front left
        1: front left-center
        2: front center
        3: front right-center
        4: front right
        5: rear left
        6: rear right

    - If right side is stronger -> force a LEFT turn (w > 0)
    - If left side is stronger  -> force a RIGHT turn (w < 0)
    """
    s = sensor_vals

    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    # nothing significant
    if left_side < side_thr and right_side < side_thr:
        return w

    if right_side > left_side and right_side >= side_thr:
        # obstacle on RIGHT -> force LEFT turn (w > 0)
        if w <= 0:
            w = min_turn
        else:
            w = max(w, min_turn)
    elif left_side > right_side and left_side >= side_thr:
        # obstacle on LEFT -> force RIGHT turn (w < 0)
        if w >= 0:
            w = -min_turn
        else:
            w = min(w, -min_turn)

    return w


def apply_center_heuristic(sensor_vals, v, w, min_turn=1.8, center_thr=1500):
    """
    Special handling when the FRONT-CENTER sensor (index 2) sees something.

    - Strong turn left/right depending on which side also sees the obstacle more.
    - Slightly reduce v so that the robot turns more aggressively and doesn't hit
      the obstacle even if it is directly in front.
    """
    s = sensor_vals
    center_val = s[2]

    if center_val < center_thr:
        return v, w

    # Decide turn direction by comparing left/right sides
    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    # Default: if both sides very small, keep the existing sign of w,
    # but enforce a minimum magnitude.
    if left_side < center_thr and right_side < center_thr:
        if w >= 0:
            w = max(w, min_turn)
        else:
            w = min(w, -min_turn)
    else:
        # Stronger reading on the right -> turn LEFT (w > 0)
        if right_side >= left_side:
            if w <= 0:
                w = min_turn
            else:
                w = max(w, min_turn)
        # Stronger reading on the left -> turn RIGHT (w < 0)
        else:
            if w >= 0:
                w = -min_turn
            else:
                w = min(w, -min_turn)

    # Reduce forward speed so the robot doesn't "push" into the obstacle
    v = min(v, 0.05)

    return v, w




async def run():
    """
    PURE OBSTACLE AVOIDANCE on a real Thymio.

    Usage in a Jupyter notebook:
        from this_file import run
        await run()
    """
    # ----- create local grid, navigator, controller -----
    grid = LocalOccupancyGrid(
        size_m=0.8,
        resolution_m=0.01,
        hit_inc=0.4,
        decay=0.85,
        robot_radius_m=0.12,
    )

    cfg = LocalNavConfig(
        occ_threshold=0.3,
        rep_influence_radius=0.8,
        k_att=1.2,
        k_rep=0.85,
    )
    navigator = LocalNavigator(grid, cfg)

    gains = GoToGoalGains(
        Komega=2.0,   # more aggressive turning
        v_max=0.35,   # faster forward motion
        w_max=4.0,    # allow faster rotation
    )
    kin = ThymioKinematics()
    g2g = GoToGoalController(gains, kin)

    # ----- connect to Thymio -----
    client = ClientAsync()
    node = await client.wait_for_node()
    await node.lock()
    await node.wait_for_variables({"prox.horizontal"})

    print("Starting PURE OBSTACLE AVOIDANCE test...")

    try:
        while True:
            # --- get sensors from Thymio ---
            sensor_vals = list(node.v.prox.horizontal)

            # --- compute forces in robot frame (updates grid internally) ---
            F_att, F_rep, F_tot = navigator.compute_avoidance_direction_robot(sensor_vals)

            # --- base (v, w) from potential field ---
            v, w = avoidance_unicycle_cmd(F_tot, gains=gains)

            # --- heuristic: outer sensors choose turning side and ensure min |w| ---
            w = apply_side_heuristic(sensor_vals, w, min_turn=1.5, side_thr=1200)

            # --- heuristic: center sensor -> VERY strong turn + smaller v ---
            v, w = apply_center_heuristic(sensor_vals, v, w, min_turn=1.8, center_thr=1500)

            # --- map unicycle -> wheel commands and send to Thymio ---
            uL, uR = g2g.unicycle_to_wheels(v, w)

            await node.set_variables({
                "motor.left.target":  [uL],
                "motor.right.target": [uR],
            })

            # --- Debug print ---
            mag_att, ang_att = vec_to_polar(F_att)
            mag_rep, ang_rep = vec_to_polar(F_rep)
            mag_tot, ang_tot = vec_to_polar(F_tot)

            print(
                f"sensors={sensor_vals} | "
                f"F_att mag={mag_att:.2f}, ang={ang_att:+.1f}° | "
                f"F_rep mag={mag_rep:.2f}, ang={ang_rep:+.1f}° | "
                f"F_tot mag={mag_tot:.2f}, ang={ang_tot:+.1f}° | "
                f"v={v:.3f}, w={w:.3f}, uL={uL}, uR={uR}"
            )

            # control loop period: smaller -> more frequent updates and faster reaction
            await client.sleep(0.07)

    finally:
        # stop motors and unlock robot even if something goes wrong
        await node.set_variables({
            "motor.left.target":  [0],
            "motor.right.target": [0],
        })
        await node.unlock()




await run()



Starting PURE OBSTACLE AVOIDANCE test...
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep mag=0.00, ang=+0.0° | F_tot mag=1.20, ang=+0.0° | v=0.350, w=0.000, uL=159, uR=159
sensors=[0, 0, 0, 0, 0, 0, 0] | F_att mag=1.20, ang=+0.0° | F_rep

CancelledError: 